In [ ]:
!pip install -q diffusers transformers accelerate

In [ ]:
import torch, json, time
from pathlib import Path
from PIL import Image, ImageFilter
from diffusers import StableDiffusionInpaintPipeline

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to("cuda")
pipe.set_progress_bar_config(disable=True)
print("loaded")

model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


loaded


In [ ]:
PROMPT = ("macro photograph of a single decaying green leaf, bug eaten holes, visible through each opening, torn brown necrotic edges, bug eaten, detailed leaf veins, plant pathology, backlight shining through holes, shallow depth of field, sharp focus, 8k resolution, organic textures ")
NEG = ""

GUIDANCE, STEPS = 11, 50

IN  = Path('/content/upload_p3')
OUT = Path('/content/p3_out')
OUT.mkdir(parents=True, exist_ok=True)

masks = sorted(IN.glob('*_mask.png'))
print(f"{len(masks)} masks")

log = []
for mp in masks:
    src = IN / f"{mp.stem.replace('_mask','')}.jpg"
    if not src.exists():
        print("missing", src.name); continue

    img  = Image.open(src).convert('RGB')
    mask = Image.open(mp).convert('L').filter(ImageFilter.GaussianBlur(2))
    seed = abs(hash(src.stem)) % (2**31)

    t = time.time()
    res = pipe(prompt=PROMPT, negative_prompt=NEG,
               image=img, mask_image=mask,
               guidance_scale=GUIDANCE, num_inference_steps=STEPS,
               generator=torch.Generator("cuda").manual_seed(seed)).images[0]

    # SD returns 512x512. Without this, resolution alone separates
    # P3 from P1 and the benchmark is worthless.
    res = res.resize(img.size, Image.LANCZOS)
    res.save(OUT / f"{src.stem}_P3.jpg", quality=95)

    log.append({"src": src.name, "mask": mp.name, "seed": seed})
    if len(log) % 25 == 0:
        print(f"  {len(log)}/{len(masks)}   {time.time()-t:.1f}s/img")

json.dump({"pool": "P3 - inpainted holes into P1 (real healthy)",
           "model": "stable-diffusion-v1-5/stable-diffusion-inpainting",
           "prompt": PROMPT, "negative": NEG,
           "guidance": GUIDANCE, "steps": STEPS, "mask_blur": 2,
           "mask_source": "auto_mask.py seed 3 scale 70",
           "dtype": "float16", "device": "cuda",
           "resize_back": "LANCZOS to source size",
           "items": log}, open(OUT/'generation_log.json','w'), indent=2)
print(f"done: {len(log)}")

198 masks
  25/198   9.6s/img
  50/198   9.6s/img
  75/198   9.6s/img
  100/198   9.6s/img
  125/198   9.6s/img
  150/198   9.6s/img
  175/198   9.6s/img
done: 198


In [ ]:
!zip -r p3_output.zip /content/p3_out

  adding: content/p3_out/ (stored 0%)
  adding: content/p3_out/healthy89__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy171__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy178__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy31__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy188__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy117__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy146__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy74__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy145__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy52__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy137__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy120__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy32__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy168__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy34__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy199__P3.jpg (deflated 0%)
  adding: content/p3_out/healthy110__P3.